In [1]:
import numpy as np
import pandas as pd
import os
import glob
import cv2
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D , MaxPooling2D , Dense , Flatten , Dropout , BatchNormalization , GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping , ModelCheckpoint , ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.model_selection import train_test_split

In [2]:
def Load_Data(Data_DIR):
  Images = []
  Labels = []
  Class = sorted(os.listdir(Data_DIR))
  Classes = len(Class)

  for Label, Folder in enumerate(Class):
    Class_DIR = os.path.join(Data_DIR, Folder)
    if not os.path.isdir(Class_DIR):
      continue

    for Img_Name in os.listdir(Class_DIR):
      Img_Path = os.path.join(Class_DIR, Img_Name)

      Img = cv2.imread(Img_Path)

      if Img is None:
        print(f'IMG Not Found or Invalid: {Img_Path}')
        continue

      Img = cv2.cvtColor(Img, cv2.COLOR_BGR2RGB)

      Img = cv2.resize(Img, (100, 100))

      Img = preprocess_input(Img)

      Images.append(Img)
      Labels.append(Label)

  return np.array(Images), np.array(Labels), Classes

In [3]:
Train_Data = r"E:\reza\program maker\Deep Learning Data's\GENDER"

In [4]:
X , y , Classes = Load_Data(Train_Data)

In [5]:
X_Train , X_Test , y_Train , y_Test = train_test_split(
    X , y , random_state=42 , test_size=0.2 , shuffle=True
)

In [6]:
DataGen = ImageDataGenerator(
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=10
)


if X_Train.ndim == 3:
    X_Train = np.expand_dims(X_Train, axis=-1)

if X_Test.ndim == 3:
    X_Test = np.expand_dims(X_Test, axis=-1)

DataGen.fit(X_Train)

In [7]:
Base = VGG16(input_shape=(100 , 100 , 3) , include_top=False , weights='imagenet' )
Base.trainable = True

for layer in Base.layers[:-4]:
  layer.trainable = False

In [8]:
Model = Sequential([
    Base ,
    GlobalAveragePooling2D(),
    Dense(256 , activation='relu' , kernel_regularizer=l2(0.001)),
    Dense(2 , activation='softmax')
])
Model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)                   │ (None, 3, 3, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 2)                   │             514 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,846,530 (56.64 MB)

 Trainable params: 7,211,266 (27.51 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [9]:
Model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:

Reduce_LR = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,       
    patience=3,       
    min_lr=1e-6,     
    verbose=1
)


Checkpoint = ModelCheckpoint(
    'Best_Gender_Model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

Early_Stopping = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

My_CallBacks = [Reduce_LR, Checkpoint, Early_Stopping]

In [11]:
History = Model.fit(
    DataGen.flow(X_Train, y_Train, batch_size=32),
    validation_data=(X_Test, y_Test),
    epochs=30,
    callbacks=My_CallBacks,
    verbose=1
)

Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6152 - loss: 3.2934
Epoch 1: val_loss improved from None to 2.41495, saving model to Best_Gender_Model.keras

Epoch 1: finished saving model to Best_Gender_Model.keras
11/11 ━━━━━━━━━━━━━━━━━━━━ 49s 4s/step - accuracy: 0.6152 - loss: 3.2934 - val_accuracy: 0.5930 - val_loss: 2.4150 - learning_rate: 1.0000e-05
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6676 - loss: 1.7698
Epoch 2: val_loss improved from 2.41495 to 1.70729, saving model to Best_Gender_Model.keras

Epoch 2: finished saving model to Best_Gender_Model.keras
11/11 ━━━━━━━━━━━━━━━━━━━━ 46s 4s/step - accuracy: 0.6676 - loss: 1.7698 - val_accuracy: 0.6628 - val_loss: 1.7073 - learning_rate: 1.0000e-05
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7493 - loss: 1.0734
Epoch 3: val_loss improved from 1.70729 to 1.41449, saving model to Best_Gender_Model.keras

Epoch 3: finished saving model to Best_Gender_Model.keras
11/11 ━━━━━━━━━